In [4]:
model = "llama3.2:1B"

#### Task 1: Create a Simple Chain for Summarization


**Objective:**

Build a LangChain chain that can summarize a given text.

**Task Description:**

- Create a llm chain using with a Ollama model.
- Define a prompt template for summarization. The summary should be only one sentence.
- Run the chain with a sample text and print the summary.
- Add model output streaming.
- Run the chain with streaming with a sample text and print the summary.


**Hint: The five messages in LangChain:**


- SystemMessage: corresponds to system role
- HumanMessage: corresponds to user role
- AIMessage: corresponds to assistant role
- AIMessageChunk: corresponds to assistant role, used for streaming responses
- ToolMessage: corresponds to tool role


[More Information](https://python.langchain.com/docs/concepts/messages/)

**Useful links:**

- [How To Prompt Template 1](https://python.langchain.com/v0.2/docs/tutorials/extraction/#the-extractor)
- [How To Prompt Template 2](https://python.langchain.com/v0.2/api_reference/core/prompts/langchain_core.prompts.chat.ChatPromptTemplate.html#langchain_core.prompts.chat.ChatPromptTemplate)
- [How To LCEL Chains 1](https://python.langchain.com/v0.2/docs/concepts/#langchain-expression-language-lcel)
- [How To LCEL Chains 2](https://python.langchain.com/v0.2/docs/versions/migrating_chains/llm_chain/#lcel)
- [How To Chain Streaming](https://python.langchain.com/v0.2/docs/concepts/#streaming)

In [5]:
from langchain_ollama.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

# Load the Ollama model
model = "llama3.2:1B"
llm = ChatOllama(model=model)

# ADD HERE YOUR CODE
# Define the prompt template
summarization_prompt = ChatPromptTemplate([
    ("system", "Du bist ein hilfreicher Assistent, der einen Text zusammenfassen soll."),
    ("human", "Fasse den folgenden Text zusammen in 1-2 Sätzen: {text}"),
])

# ADD HERE YOUR CODE
# Create the LLMChain
summarization_chain = summarization_prompt | llm

# Sample text
text = """Over the last decade, deep learning has evolved massively to process and generate unstructured data like text, images, and video. 
These advanced AI models have gained popularity in various industries, and include large language models (LLMs). 
There is currently a significant level of fanfare in both the media and the industry surrounding AI,
and there’s a fair case to be made that Artificial Intelligence (AI), with these advancements,
is about to have a wide-ranging and major impact on businesses, societies, and individuals alike.
This is driven by numerous factors, including advancements in technology, high-profile applications, 
and the potential for transfor- mative impacts across multiple sectors."""

# ADD HERE YOUR CODE
# Invoke the chain
summary = summarization_chain.invoke({"text": text})
print(summary.content)

Deep learning technologies haben sich in den letzten Jahren enorm entwickelt und sind nun in vielen Branchen wie großen Sprachmodellen (LLMs) eingesetzt. Mit diesen Fortschritten ist es scheinbar wahrscheinlich, dass AI einen breiten Einfluss auf Unternehmen, Gesellschaften und Individuen haben wird.


In [6]:
# Stream the chain output
for chunk in summarization_chain.stream({"text": text}):
    print(chunk.content, end="", flush=True)

Deep learning hat sich im letzten Jahrzehnt massiv entwickelt und wird nun verwendet, um un structures Daten wie Texte, Bilder und Videos zu verarbeiten.  AI ist mit diesen Fortschritten auf dem Höhepunkt der Populärkeits, sowohl in der Medien als auch in der Branche, da es viele wünschenswerte Auswirkungen auf Unternehmen, Gesellschaften und Individuen haben könnte.

#### Task 2: Chain with Tool Usage (Simple Math Tool)

**Objective:**

Create a LangChain chain that uses a simple math tool to perform calculations.

**Task Description:**

- Define a function as tool which multiplies two integer values and return the result.
- Create a chain
- Print the result of the calculation.

**Useful links:**

- [How To Tools](https://python.langchain.com/v0.2/docs/how_to/tools_chain/#create-a-tool)
- [How To Tools in Chains](https://python.langchain.com/v0.2/docs/how_to/tools_chain/#chains)
- [How To Tool Calling](https://python.langchain.com/v0.2/docs/concepts/#functiontool-calling)
- [How To Chain and Call Tools with Ollama](https://python.langchain.com/v0.2/docs/integrations/chat/ollama/)

In [7]:
from langchain_core.tools import tool

# ADD HERE YOUR CODE
# Create custom tool
@tool
def multiply(first_int: int, second_int: int) -> int:
    """Multipliziere die gegebene Zahl mit 2."""
    return first_int * second_int


print(multiply.name)
print(multiply.description)
print(multiply.args) # -> definition of tool arguments

# Invoke custom tool
multiply.invoke({"first_int": 4, "second_int": 5})

multiply
Multipliziere die gegebene Zahl mit 2.
{'first_int': {'title': 'First Int', 'type': 'integer'}, 'second_int': {'title': 'Second Int', 'type': 'integer'}}


20

In [8]:
# Load the Ollama model
llm = ChatOllama(model=model)

# ADD HERE YOUR CODE
# Use bind_tools to pass the definition of our tool in as part of each call to the model, so that the model can invoke the tool
llm_with_tools = llm.bind_tools([multiply])

# When the model invokes the tool, this will show up in the AIMessage.tool_calls attribute of the output -> extract tool parameters from input text
msg = llm_with_tools.invoke("whats 5 times forty two")
print(msg.tool_calls)

[{'name': 'multiply', 'args': {'first_int': '42', 'second_int': '5'}, 'id': '813faebe-13fc-4fcd-a28e-0e945ad2a77b', 'type': 'tool_call'}]


In [9]:
# ADD HERE YOUR CODE
# Create the chain: pass the extracte tool parameters from the input text to the tool -> extract the arguments of the first tool_call
chain_with_tools = llm.bind_tools([multiply])

# Run chain
chain_with_tools.invoke("whats 5 times forty two")

AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:1B', 'created_at': '2026-04-08T18:28:12.4871535Z', 'done': True, 'done_reason': 'stop', 'total_duration': 519210700, 'load_duration': 187793100, 'prompt_eval_count': 166, 'prompt_eval_duration': 9485700, 'eval_count': 33, 'eval_duration': 288808900, 'logprobs': None, 'model_name': 'llama3.2:1B', 'model_provider': 'ollama'}, id='lc_run--019d6e5a-1a3f-7083-83d1-5e88c7a8fdff-0', tool_calls=[{'name': 'multiply', 'args': {'first_int': 42, 'second_int': 5}, 'id': '75a71433-14c5-4bc7-9cec-e3287b8ff72b', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 166, 'output_tokens': 33, 'total_tokens': 199})

#### Task 3: Agent with Tool Usage (Two Tools)

**Objective:** 

Create a LangChain agent that uses two tools to perform tasks.

**Task Description:**

- Define the system prompt.
- Define tools.
- Create an Agent using the Ollama model, system-prompt and tools.
- Run the agent with a prompt that requires one or both tools.
- Observe how the agent uses the tools to complete the task.


**Useful links:**

- [How To](https://docs.langchain.com/oss/javascript/langchain/agents)

In [10]:
from langchain.agents import create_agent

# Load the Ollama model
llm = ChatOllama(model=model)

# ADD HERE YOUR CODE
agent_prompt = """Du bist ein hilfreicher Assistent, der einfache mathematische Aufgaben lösen soll."""

# Custom math tools
@tool
def add(first_int: int, second_int: int) -> int:
    """Add two integers."""
    return first_int + second_int


@tool
def exponentiate(base: int, exponent: int) -> int:
    """Exponentiate the base to the exponent power."""
    return base**exponent


tools = [add, exponentiate]

In [11]:
# ADD HERE YOUR CODE
# Construct the agent
agent_with_tools = create_agent(llm, tools)

In [ ]:
response = agent_with_tools.invoke(                                                                             # Ruft den Agent mit Input auf
    {"messages": [{"role": "user", "content": "First take 3 to the power of five and afterwards add 12?"}]}     # Übergabe im Chat format
)

# Get the last AI message
ai_messages = [m for m in response["messages"] if m.type == "ai"]                                               # Filtert nur die AI-Nachrichten aus der Antwort heraus
if ai_messages:
    last_message = ai_messages[-1]
    print(last_message.content)                                                                                 # finale Antwort

You received a Tool Call Response!

To format your original question, I'll take 3 to the power of five and add 12.

"Three to the power of five is 243. Adding 12 gives us 255."
